#### Libraries

In [2]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import random
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold

# Plotting style
sns.set_style('darkgrid')
sns.set_theme(font_scale=1.)

# Set the state (we will call the function when we need to make sure that we get the same results every time we run the code)
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_all_seeds(1234)

In [ ]:
def get_basic_model(input_dim, hidden_dim, output_dim):
    # Assignment: Define the model usign the torch.nn.Sequential approach (see the exercise part for help)
    # YOUR CODE HERE
    return torch.nn.Sequential(
        torch.nn.Linear(in_features=input_dim, out_features=hidden_dim, bias=True),     # Input layer
        torch.nn.ReLU(),                                                                # Activation function
        torch.nn.Linear(in_features=hidden_dim, out_features=output_dim, bias=True),    # Output layer
    ) 

# Here we call a specific instance of the model
hidden_dim = 10
input_dim = 6
output_dim = 10

model = get_basic_model(input_dim, hidden_dim, output_dim)
print(f"Model: {model}")

In [ ]:
def fit_nn(model, X_train, y_train, X_test, y_test, criterion,n_epochs,lr):
    # Assignment: Define the 'optimizer' as stochastic gradient descent (SGD) method with the learning rate defined in lr.     
    optimizer = torch.optim.SGD(params=model.parameters(), lr=lr)
    # YOUR CODE HERE
    

    # Define a dictionary to store the loss values for each epoch
    results = {'train': [], 'test': []}

    # Training loop
    for epoch in range(n_epochs):    
        # Set the model to training mode
        model.train()
        # Make sure that the gradients are zero before you use backpropagation
        optimizer.zero_grad()
        # Make a forward pass through the model to compute the outputs
        outputs = model(X_train)
        # Compute the loss
        loss = criterion(outputs, y_train)
        # Do a backward pass to compute the gradients wrt. model parameters using backpropagation.
        loss.backward()
        # Update the model parameters by making the optimizer take a gradient descent step
        optimizer.step()

        with torch.no_grad(): # No need to compute gradients for the validation set
            # Set the model to evaluation mode
            model.eval()  
            # Compute the loss for the test set
            test_outputs = model(X_test)
            test_loss = criterion(test_outputs, y_test)
        
        # Store the training and test loss for this epoch in the dictionary
        results['train'].append(loss.item())
        results['test'].append(test_loss.item())

        # Print the loss every 1000 epochs
        if (epoch+1) % 1000 == 0:
            print(f'Epoch [{epoch+1}/{n_epochs}], Training loss: {loss.item():.4f}, Test Loss: {test_loss.item():.4f}')

    return model, results

In [ ]:
df = pd.read_csv('HR_data.csv')

X = df.drop(columns=['Cohort', 'Round', 'Frustrated'])
X = X.iloc[:, 1:]
X['Phase'] = X['Phase'].map({'phase1': 1, 'phase2': 2, 'phase3': 3})
y = df['Frustrated']

print(X.shape)
X

,HR_Mean,HR_Median,HR_std,HR_Min,HR_Max,HR_AUC,Phase,Individual,Puzzler
0,77.965186,78.000,3.345290,73.23,83.37,22924.945,3,1,1
1,70.981097,70.570,2.517879,67.12,78.22,21930.400,2,1,1
2,73.371959,73.360,3.259569,67.88,80.22,21647.085,1,1,1
3,78.916822,77.880,4.054595,72.32,84.92,25258.905,3,1,1
4,77.322226,74.550,6.047603,70.52,90.15,23890.565,2,1,1
...,...,...,...,...,...,...,...,...,...
163,73.594539,72.380,9.474556,57.43,93.53,21482.985,2,14,0
164,57.839897,54.130,6.796647,52.97,74.14,16825.740,1,14,0
165,64.237295,65.195,3.589241,58.97,72.63,18691.065,3,14,0
166,70.834320,70.440,2.391160,66.65,76.07,20753.005,2,14,0


In [ ]:
criterion = torch.nn.CrossEntropyLoss()